In [11]:
import cv2
import mediapipe as mp
import os
import numpy as np

# Ruta a tu base de datos de imágenes
DATA_DIR = 'data/db'

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True)
labels = ['A', 'B', 'C', 'D', 'E']

X, y = [], []

for idx, label in enumerate(labels):
    folder_path = os.path.join(DATA_DIR, label)
    for filename in os.listdir(folder_path):
        if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
        img_path = os.path.join(folder_path, filename)
        img = cv2.imread(img_path)
        if img is None:
            continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        results = hands.process(img_rgb)
        if results.multi_hand_landmarks:
            landmarks = []
            for lm in results.multi_hand_landmarks[0].landmark:
                landmarks.extend([lm.x, lm.y, lm.z])
            X.append(landmarks)
            y.append(idx)
        # Si quieres saber cuántas imágenes no detectó MediaPipe:
        # else:
        #     print(f"No hand detected in {img_path}")

X = np.array(X)
y = np.array(y)

# Guarda los datos procesados para usarlos después
np.save('X_hand_sign_vowels.npy', X)
np.save('y_hand_sign_vowels.npy', y)

print("Listo. Keypoints extraídos:", X.shape)


Listo. Keypoints extraídos: (1045, 63)


In [12]:
import numpy as np
from tensorflow import keras
from sklearn.model_selection import train_test_split

# Cargar los datos
X = np.load('X_hand_sign_vowels.npy')
y = np.load('y_hand_sign_vowels.npy')

# One-hot encoding para las salidas (5 clases: A-E)
y_cat = keras.utils.to_categorical(y, num_classes=5)

# División train/test
X_train, X_test, y_train, y_test = train_test_split(X, y_cat, test_size=0.2, random_state=42)

# Modelo
model = keras.Sequential([
    keras.layers.Input(shape=(63,)), # 21 puntos x 3 coordenadas
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(5, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=30, batch_size=16)

# Guardar modelo
model.save('hand_sign_vowels_model.h5')


Epoch 1/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.4341 - loss: 1.4456 - val_accuracy: 0.6411 - val_loss: 1.0229
Epoch 2/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7048 - loss: 0.9326 - val_accuracy: 0.8373 - val_loss: 0.6183
Epoch 3/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8640 - loss: 0.5854 - val_accuracy: 0.8565 - val_loss: 0.4412
Epoch 4/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8869 - loss: 0.4150 - val_accuracy: 0.9139 - val_loss: 0.3143
Epoch 5/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9179 - loss: 0.3257 - val_accuracy: 0.9378 - val_loss: 0.2660
Epoch 6/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9354 - loss: 0.2703 - val_accuracy: 0.8756 - val_loss: 0.3045
Epoch 7/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9263 - loss: 0.2583 - val_accuracy: 0.8947 - val_loss: 0.2254
Epoch 8/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9547 - loss: 0.1931 - val_accuracy: 0.9426 - val_loss

In [13]:
import tensorflow as tf

model = tf.keras.models.load_model('hand_sign_vowels_model.h5')
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('hand_sign_vowels_model.tflite', 'wb') as f:
    f.write(tflite_model)

print("Modelo .tflite generado.")


INFO:tensorflow:Assets written to: C:\Users\Daniel\AppData\Local\Temp\tmpqwp1hdvc\assets


INFO:tensorflow:Assets written to: C:\Users\Daniel\AppData\Local\Temp\tmpqwp1hdvc\assets


Saved artifact at 'C:\Users\Daniel\AppData\Local\Temp\tmpqwp1hdvc'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 63), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  2188028360656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2188028361040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2188028347408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2188028358736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2188028360080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2188028356432: TensorSpec(shape=(), dtype=tf.resource, name=None)
Modelo .tflite generado.


In [4]:
pip install opencv-python mediapipe tensorflow


Note: you may need to restart the kernel to use updated packages.


In [14]:
import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf

# Carga el modelo TFLite
interpreter = tf.lite.Interpreter(model_path="hand_sign_vowels_model.tflite")
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Inicializa MediaPipe Hands
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.7)
mp_drawing = mp.solutions.drawing_utils

# Labels
labels = ['A', 'B', 'C', 'D', 'E']

# Captura de cámara
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)
    h, w, _ = frame.shape
    prediction_text = ""
    
    if results.multi_hand_landmarks:
        hand_landmarks = results.multi_hand_landmarks[0]
        # Extraer y normalizar los keypoints (en relación al tamaño de la imagen)
        keypoints = []
        for lm in hand_landmarks.landmark:
            keypoints.extend([lm.x, lm.y, lm.z])
        keypoints = np.array(keypoints, dtype=np.float32).reshape(1, -1)

        # Si tienes menos de 63 valores, completa con ceros
        if keypoints.shape[1] < 63:
            keypoints = np.pad(keypoints, ((0,0),(0,63-keypoints.shape[1])), 'constant')
        
        # Ejecutar inferencia TFLite
        interpreter.set_tensor(input_details[0]['index'], keypoints)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]['index'])
        pred_idx = np.argmax(output)
        confidence = output[0][pred_idx]
        prediction_text = f"{labels[pred_idx]} ({confidence:.2f})"

        # Dibuja la mano
        mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

    # Muestra la predicción
    cv2.putText(frame, f"Prediccion: {prediction_text}", (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2, cv2.LINE_AA)

    cv2.imshow("Hand Sign Recognition", frame)
    if cv2.waitKey(1) & 0xFF == 27: # ESC para salir
        break

cap.release()
cv2.destroyAllWindows()


c:\Users\Daniel\Downloads\EspSignLangModel-main\EspSignLangModel-main\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
